# Re-auditing the compliance operator under `wall=:clamped`

`derivations/audit_compliance_operator.jl` established, for the free wall (`wall=:free`), that
the pressure-to-gap compliance operator is asymmetric only through the droplet block (a
radial-vs-vertical projection mismatch, not a bug -- documented, not fixed, since the
Newton solve never assumes symmetry), and that its symmetric part is positive semi-definite
up to a numerical noise floor set by the operator's own compactness.

Volume-conserving pinning (`wall=:clamped`, `apply_clamp` in `src/residual.jl`) changes the
bath's response to pressure by a rank-one correction (derived in
`derivations/audit_compliance_operator_core.jl`'s `compliance` docstring). A rank-one update
can just as easily break self-adjointness or definiteness as leave them alone -- this
notebook checks which, rather than assuming the free-wall proof carries over unchanged.

In [1]:
import Pkg
Pkg.activate(joinpath(@__DIR__, ".."))
using SpectralKM
using SpectralKM: gauss_legendre_nodes, legendre_P_table, apply_clamp
using SpecialFunctions: besselj0
using LinearAlgebra, Printf, Base64
include(joinpath(@__DIR__, "..", "derivations", "audit_compliance_operator_core.jl"))
println("ready")

  Activating 

ready


project at `~/Documents/Github/1pkm-drop-onto-bath`


In [2]:
function svgplot(series; width=680, height=300, xlabel="", ylabel="", title="", xlim=nothing, ylim=nothing)
    pad = 52
    xs = vcat((s.x for s in series)...); ys = vcat((s.y for s in series)...)
    x0, x1 = xlim === nothing ? (minimum(xs), maximum(xs)) : xlim
    y0, y1 = ylim === nothing ? (minimum(ys), maximum(ys)) : ylim
    x1 == x0 && (x1 = x0 + 1); y1 == y0 && (y1 = y0 + 1)
    sx = v -> pad + (v - x0) / (x1 - x0) * (width - 1.6pad)
    sy = v -> height - pad - (v - y0) / (y1 - y0) * (height - 1.7pad)
    io = IOBuffer()
    print(io, "<svg xmlns=\'http://www.w3.org/2000/svg\' width=\'$width\' height=\'$height\' font-family=\'sans-serif\' font-size=\'12\'>")
    print(io, "<rect width=\'$width\' height=\'$height\' fill=\'white\'/>")
    for frac in 0:0.25:1
        xv = x0 + frac * (x1 - x0); yv = y0 + frac * (y1 - y0)
        print(io, "<line x1=\'$(sx(xv))\' y1=\'$(height-pad)\' x2=\'$(sx(xv))\' y2=\'$(height-pad+5)\' stroke=\'black\'/>")
        print(io, "<text x=\'$(sx(xv))\' y=\'$(height-pad+18)\' text-anchor=\'middle\'>$(round(xv,sigdigits=3))</text>")
        print(io, "<line x1=\'$pad\' y1=\'$(sy(yv))\' x2=\'$(pad-5)\' y2=\'$(sy(yv))\' stroke=\'black\'/>")
        print(io, "<text x=\'$(pad-9)\' y=\'$(sy(yv)+4)\' text-anchor=\'end\'>$(round(yv,sigdigits=3))</text>")
    end
    print(io, "<line x1=\'$pad\' y1=\'$(height-pad)\' x2=\'$(width-0.6pad)\' y2=\'$(height-pad)\' stroke=\'black\'/>")
    print(io, "<line x1=\'$pad\' y1=\'$(height-pad)\' x2=\'$pad\' y2=\'$(pad*0.5)\' stroke=\'black\'/>")
    for s in series
        pts = join(("$(sx(s.x[i])),$(sy(s.y[i]))" for i in eachindex(s.x)), " ")
        w = get(s, :width, 1.8)
        print(io, "<polyline points=\'$pts\' fill=\'none\' stroke=\'$(s.color)\' stroke-width=\'$w\'/>")
        for i in eachindex(s.x)
            print(io, "<circle cx=\'$(sx(s.x[i]))\' cy=\'$(sy(s.y[i]))\' r=\'3\' fill=\'$(s.color)\'/>")
        end
    end
    for (i, s) in enumerate(series)
        haskey(s, :label) || continue
        print(io, "<line x1=\'$(width-190)\' y1=\'$(pad*0.5+14i)\' x2=\'$(width-165)\' y2=\'$(pad*0.5+14i)\' stroke=\'$(s.color)\' stroke-width=\'2.4\'/>")
        print(io, "<text x=\'$(width-159)\' y=\'$(pad*0.5+14i+4)\'>$(s.label)</text>")
    end
    print(io, "<text x=\'$(width/2)\' y=\'$(height-8)\' text-anchor=\'middle\'>$xlabel</text>")
    print(io, "<text x=\'14\' y=\'$(height/2)\' text-anchor=\'middle\' transform=\'rotate(-90 14 $(height/2))\'>$ylabel</text>")
    print(io, "<text x=\'$(width/2)\' y=\'18\' text-anchor=\'middle\' font-size=\'14\'>$title</text></svg>")
    return HTML(String(take!(io)))
end

function svgbars(groups, series; width=680, height=300, ylabel="", title="", ylim=nothing)
    pad = 52; ng = length(groups); ns = length(series)
    vals = vcat((s.y for s in series)...)
    y0, y1 = ylim === nothing ? (min(0.0, minimum(vals)), maximum(vals)) : ylim
    y1 == y0 && (y1 = y0 + 1)
    sy = v -> height - pad - (v - y0) / (y1 - y0) * (height - 1.7pad)
    gw = (width - 1.6pad) / ng
    bw = gw / (ns + 1)
    io = IOBuffer()
    print(io, "<svg xmlns=\'http://www.w3.org/2000/svg\' width=\'$width\' height=\'$height\' font-family=\'sans-serif\' font-size=\'12\'>")
    print(io, "<rect width=\'$width\' height=\'$height\' fill=\'white\'/>")
    print(io, "<line x1=\'$pad\' y1=\'$(sy(0))\' x2=\'$(width-0.6pad)\' y2=\'$(sy(0))\' stroke=\'black\'/>")
    for (gi, g) in enumerate(groups)
        x0 = pad + (gi - 1) * gw
        print(io, "<text x=\'$(x0+gw/2)\' y=\'$(height-pad+18)\' text-anchor=\'middle\'>$g</text>")
        for (si, s) in enumerate(series)
            v = s.y[gi]
            xb = x0 + si * bw
            ytop = sy(max(v, 0.0)); ybot = sy(min(v, 0.0))
            print(io, "<rect x=\'$xb\' y=\'$ytop\' width=\'$(bw*0.85)\' height=\'$(max(ybot-ytop,0.5))\' fill=\'$(s.color)\'/>")
        end
    end
    for (si, s) in enumerate(series)
        print(io, "<line x1=\'$(width-190)\' y1=\'$(pad*0.5+14si)\' x2=\'$(width-165)\' y2=\'$(pad*0.5+14si)\' stroke=\'$(s.color)\' stroke-width=\'8\'/>")
        print(io, "<text x=\'$(width-159)\' y=\'$(pad*0.5+14si+4)\'>$(s.label)</text>")
    end
    print(io, "<text x=\'14\' y=\'$(height/2)\' text-anchor=\'middle\' transform=\'rotate(-90 14 $(height/2))\'>$ylabel</text>")
    print(io, "<text x=\'$(width/2)\' y=\'18\' text-anchor=\'middle\' font-size=\'14\'>$title</text></svg>")
    return HTML(String(take!(io)))
end

struct HTML s::String end
Base.show(io::IO, ::MIME"text/html", h::HTML) = print(io, h.s)
println("ready")

ready


## Self-adjointness: does pinning change the asymmetry?

`total` asymmetry (the operator as a whole) and `bath` asymmetry (the block clamping actually
touches) at four representative contact angles, `:clamped` vs `:free`, same `M=L=60, N=8, nq=40`.

In [3]:
p = Params(We=1.0958, Bo=0.017, Oh=0.006, M=60, L=60, N=8, b=6.0, h0=3.0, nq=40, wall=:clamped)
beta0 = zeros(p.L + 1)
delta = 1e-3
thetas = (0.05, 0.15, 0.3, 0.6)

asym = Dict(w => Float64[] for w in (:clamped, :free))
asym_bath = Dict(w => Float64[] for w in (:clamped, :free))
for tc in thetas, wall in (:clamped, :free)
    A, Ab, _, _, w, x, om = compliance(p, tc, beta0, delta; wall=wall)
    push!(asym[wall], asymmetry(A, w, om))
    push!(asym_bath[wall], asymmetry(Ab, w, om))
end

display(svgplot([(x=collect(thetas), y=log10.(asym[:clamped]), color="#1f77b4", label="clamped, total"),
                  (x=collect(thetas), y=log10.(asym[:free]), color="#d62728", label="free, total")];
                 xlabel="theta_c", ylabel="log10(relative asymmetry)", title="Total operator asymmetry vs theta_c"))

reldiff = @. 100 * abs(asym[:clamped] - asym[:free]) / asym[:free]
display(svgbars(["θ_c=$tc" for tc in thetas], [(y=reldiff, color="#2ca02c", label="|clamped-free|/free, %")];
                 ylabel="% difference from free wall", title="How much does pinning change the total asymmetry?"))
@printf("Bath-block asymmetry (should be ~machine zero for both): clamped max=%.2e, free max=%.2e\n",
        maximum(asym_bath[:clamped]), maximum(asym_bath[:free]))

HTML("<svg xmlns='http://www.w3.org/2000/svg' width='680' height='300' font-family='sans-serif' font-size='12'><rect width='680' height='300' fill='white'/><line x1='52.0' y1='248' x2='52.0' y2='253' stroke='black'/><text x='52.0' y='266' text-anchor='middle'>0.05</text><line x1='52' y1='248.0' x2='47' y2='248.0' stroke='black'/><text x='43' y='252.0' text-anchor='end'>-3.55</text><line x1='201.20000000000002' y1='248' x2='201.20000000000002' y2='253' stroke='black'/><text x='201.20000000000002' y='266' text-anchor='middle'>0.188</text><line x1='52' y1='195.10000000000002' x2='47' y2='195.10000000000002' stroke='black'/><text x='43' y='199.10000000000002' text-anchor='end'>-3.18</text><line x1='350.4' y1='248' x2='350.4' y2='253' stroke='black'/><text x='350.4' y='266' text-anchor='middle'>0.325</text><line x1='52' y1='142.20000000000002' x2='47' y2='142.20000000000002' stroke='black'/><text x='43' y='146.20000000000002' text-anchor='end'>-2.81</text><line x1='499.59999999999997' y1='248' x2='499.59999999999997' y2='253' stroke='black'/><text x='499.59999999999997' y='266' text-anchor='middle'>0.462</text><line x1='52' y1='89.29999999999995' x2='47' y2='89.29999999999995' stroke='black'/><text x='43' y='93.29999999999995' text-anchor='end'>-2.45</text><line x1='648.8' y1='248' x2='648.8' y2='253' stroke='black'/><text x='648.8' y='266' text-anchor='middle'>0.6</text><line x1='52' y1='36.39999999999998' x2='47' y2='36.39999999999998' stroke='black'/><text x='43' y='40.39999999999998' text-anchor='end'>-2.08</text><line x1='52' y1='248' x2='648.8' y2='248' stroke='black'/><line x1='52' y1='248' x2='52' y2='26.0' stroke='black'/><polyline points='52.0,247.99557098976447 160.5090909090909,127.86693071501662 323.27272727272725,79.68726852308131 648.8,36.39999999999998' fill='none' stroke='#1f77b4' stroke-width='1.8'/><circle cx='52.0' cy='247.99557098976447' r='3' fill='#1f77b4'/><circle cx='160.5090909090909' cy='127.86693071501662' r='3' fill='#1f77b4'/><circle cx='323.27272727272725' cy='79.68726852308131' r='3' fill='#1f77b4'/><circle cx='648.8' cy='36.39999999999998' r='3' fill='#1f77b4'/><polyline points='52.0,248.0 160.5090909090909,127.87108104345442 323.27272727272725,79.69129686184002 648.8,36.404202921893756' fill='none' stroke='#d62728' stroke-width='1.8'/><circle cx='52.0' cy='248.0' r='3' fill='#d62728'/><circle cx='160.5090909090909' cy='127.87108104345442' r='3' fill='#d62728'/><circle cx='323.27272727272725' cy='79.69129686184002' r='3' fill='#d62728'/><circle cx='648.8' cy='36.404202921893756' r='3' fill='#d62728'/><line x1='490' y1='40.0' x2='515' y2='40.0' stroke='#1f77b4' stroke-width='2.4'/><text x='521' y='44.0'>clamped, total</text><line x1='490' y1='54.0' x2='515' y2='54.0' stroke='#d62728' stroke-width='2.4'/><text x='521' y='58.0'>free, total</text><text x='340.0' y='292' text-anchor='middle'>theta_c</text><text x='14' y='150.0' text-anchor='middle' transform='rotate(-90 14 150.0)'>log10(relative asymmetry)</text><text x='340.0' y='18' text-anchor='middle' font-size='14'>Total operator asymmetry vs theta_c</text></svg>")

HTML("<svg xmlns='http://www.w3.org/2000/svg' width='680' height='300' font-family='sans-serif' font-size='12'><rect width='680' height='300' fill='white'/><line x1='52' y1='248.0' x2='648.8' y2='248.0' stroke='black'/><text x='126.6' y='266' text-anchor='middle'>θ_c=0.05</text><rect x='126.6' y='36.39999999999998' width='63.41' height='211.60000000000002' fill='#2ca02c'/><text x='275.79999999999995' y='266' text-anchor='middle'>θ_c=0.15</text><rect x='275.79999999999995' y='49.71472040871106' width='63.41' height='198.28527959128894' fill='#2ca02c'/><text x='425.0' y='266' text-anchor='middle'>θ_c=0.3</text><rect x='425.0' y='55.543063917666046' width='63.41' height='192.45693608233395' fill='#2ca02c'/><text x='574.1999999999999' y='266' text-anchor='middle'>θ_c=0.6</text><rect x='574.1999999999999' y='47.201940974877374' width='63.41' height='200.79805902512263' fill='#2ca02c'/><line x1='490' y1='40.0' x2='515' y2='40.0' stroke='#2ca02c' stroke-width='8'/><text x='521' y='44.0'>|clamped-free|/free, %</text><text x='14' y='150.0' text-anchor='middle' transform='rotate(-90 14 150.0)'>% difference from free wall</text><text x='340.0' y='18' text-anchor='middle' font-size='14'>How much does pinning change the total asymmetry?</text></svg>")

Bath-block asymmetry (should be ~machine zero for both): clamped max=1.27e-16, free max=1.26e-16


## Definiteness: is the symmetric part still positive semi-definite?

The compliance operator is compact, so most eigenvalues of a length-`nq` discretization sit
near the noise floor regardless of wall type; what matters is whether pinning pushes any
eigenvalue to a magnitude that is negative and *not* noise -- i.e. comparable to the largest
eigenvalue, not orders of magnitude below it.

In [4]:
minev = Dict(w => Float64[] for w in (:clamped, :free))
maxev = Dict(w => Float64[] for w in (:clamped, :free))
negfrac = Dict(w => Float64[] for w in (:clamped, :free))
for tc in thetas, wall in (:clamped, :free)
    A, _, _, _, w, x, om = compliance(p, tc, beta0, delta; wall=wall)
    n = length(x)
    S = [w[i] * (-A[i, j]) / om[j] for i in 1:n, j in 1:n]
    ev = eigvals(Symmetric((S + S') / 2))
    push!(minev[wall], minimum(ev)); push!(maxev[wall], maximum(ev))
    push!(negfrac[wall], 100 * count(<(0), ev) / n)
end

display(svgplot([(x=collect(thetas), y=log10.(abs.(minev[:clamped]) ./ maxev[:clamped]), color="#1f77b4", label="clamped"),
                  (x=collect(thetas), y=log10.(abs.(minev[:free]) ./ maxev[:free]), color="#d62728", label="free")];
                 xlabel="theta_c", ylabel="log10(|min eig| / max eig)", title="Most-negative eigenvalue, relative to the largest"))

display(svgbars(["θ_c=$tc" for tc in thetas],
                 [(y=negfrac[:clamped], color="#1f77b4", label="clamped"), (y=negfrac[:free], color="#d62728", label="free")];
                 ylabel="% of eigenvalues negative", title="Fraction of the spectrum that is negative"))
println("If the two colors track each other above, pinning has not introduced any real")
println("indefiniteness -- both are sitting on the same compactness-driven noise floor.")

HTML("<svg xmlns='http://www.w3.org/2000/svg' width='680' height='300' font-family='sans-serif' font-size='12'><rect width='680' height='300' fill='white'/><line x1='52.0' y1='248' x2='52.0' y2='253' stroke='black'/><text x='52.0' y='266' text-anchor='middle'>0.05</text><line x1='52' y1='248.0' x2='47' y2='248.0' stroke='black'/><text x='43' y='252.0' text-anchor='end'>-16.0</text><line x1='201.20000000000002' y1='248' x2='201.20000000000002' y2='253' stroke='black'/><text x='201.20000000000002' y='266' text-anchor='middle'>0.188</text><line x1='52' y1='195.10000000000002' x2='47' y2='195.10000000000002' stroke='black'/><text x='43' y='199.10000000000002' text-anchor='end'>-13.5</text><line x1='350.4' y1='248' x2='350.4' y2='253' stroke='black'/><text x='350.4' y='266' text-anchor='middle'>0.325</text><line x1='52' y1='142.20000000000002' x2='47' y2='142.20000000000002' stroke='black'/><text x='43' y='146.20000000000002' text-anchor='end'>-11.1</text><line x1='499.59999999999997' y1='248' x2='499.59999999999997' y2='253' stroke='black'/><text x='499.59999999999997' y='266' text-anchor='middle'>0.462</text><line x1='52' y1='89.29999999999998' x2='47' y2='89.29999999999998' stroke='black'/><text x='43' y='93.29999999999998' text-anchor='end'>-8.63</text><line x1='648.8' y1='248' x2='648.8' y2='253' stroke='black'/><text x='648.8' y='266' text-anchor='middle'>0.6</text><line x1='52' y1='36.39999999999998' x2='47' y2='36.39999999999998' stroke='black'/><text x='43' y='40.39999999999998' text-anchor='end'>-6.18</text><line x1='52' y1='248' x2='648.8' y2='248' stroke='black'/><line x1='52' y1='248' x2='52' y2='26.0' stroke='black'/><polyline points='52.0,241.64213755681448 160.5090909090909,242.27463488187286 323.27272727272725,137.47077011348753 648.8,36.39999999999998' fill='none' stroke='#1f77b4' stroke-width='1.8'/><circle cx='52.0' cy='241.64213755681448' r='3' fill='#1f77b4'/><circle cx='160.5090909090909' cy='242.27463488187286' r='3' fill='#1f77b4'/><circle cx='323.27272727272725' cy='137.47077011348753' r='3' fill='#1f77b4'/><circle cx='648.8' cy='36.39999999999998' r='3' fill='#1f77b4'/><polyline points='52.0,245.2679143933826 160.5090909090909,248.0 323.27272727272725,137.47212601835463 648.8,36.40108422713013' fill='none' stroke='#d62728' stroke-width='1.8'/><circle cx='52.0' cy='245.2679143933826' r='3' fill='#d62728'/><circle cx='160.5090909090909' cy='248.0' r='3' fill='#d62728'/><circle cx='323.27272727272725' cy='137.47212601835463' r='3' fill='#d62728'/><circle cx='648.8' cy='36.40108422713013' r='3' fill='#d62728'/><line x1='490' y1='40.0' x2='515' y2='40.0' stroke='#1f77b4' stroke-width='2.4'/><text x='521' y='44.0'>clamped</text><line x1='490' y1='54.0' x2='515' y2='54.0' stroke='#d62728' stroke-width='2.4'/><text x='521' y='58.0'>free</text><text x='340.0' y='292' text-anchor='middle'>theta_c</text><text x='14' y='150.0' text-anchor='middle' transform='rotate(-90 14 150.0)'>log10(|min eig| / max eig)</text><text x='340.0' y='18' text-anchor='middle' font-size='14'>Most-negative eigenvalue, relative to the largest</text></svg>")

HTML("<svg xmlns='http://www.w3.org/2000/svg' width='680' height='300' font-family='sans-serif' font-size='12'><rect width='680' height='300' fill='white'/><line x1='52' y1='248.0' x2='648.8' y2='248.0' stroke='black'/><text x='126.6' y='266' text-anchor='middle'>θ_c=0.05</text><rect x='101.73333333333332' y='48.15555555555554' width='42.273333333333326' height='199.84444444444446' fill='#1f77b4'/><rect x='151.46666666666664' y='36.39999999999998' width='42.273333333333326' height='211.60000000000002' fill='#d62728'/><text x='275.79999999999995' y='266' text-anchor='middle'>θ_c=0.15</text><rect x='250.9333333333333' y='71.66666666666663' width='42.273333333333326' height='176.33333333333337' fill='#1f77b4'/><rect x='300.66666666666663' y='71.66666666666663' width='42.273333333333326' height='176.33333333333337' fill='#d62728'/><text x='425.0' y='266' text-anchor='middle'>θ_c=0.3</text><rect x='400.1333333333333' y='59.9111111111111' width='42.273333333333326' height='188.0888888888889' fill='#1f77b4'/><rect x='449.8666666666666' y='83.42222222222219' width='42.273333333333326' height='164.5777777777778' fill='#d62728'/><text x='574.1999999999999' y='266' text-anchor='middle'>θ_c=0.6</text><rect x='549.3333333333333' y='106.93333333333334' width='42.273333333333326' height='141.06666666666666' fill='#1f77b4'/><rect x='599.0666666666666' y='106.93333333333334' width='42.273333333333326' height='141.06666666666666' fill='#d62728'/><line x1='490' y1='40.0' x2='515' y2='40.0' stroke='#1f77b4' stroke-width='8'/><text x='521' y='44.0'>clamped</text><line x1='490' y1='54.0' x2='515' y2='54.0' stroke='#d62728' stroke-width='8'/><text x='521' y='58.0'>free</text><text x='14' y='150.0' text-anchor='middle' transform='rotate(-90 14 150.0)'>% of eigenvalues negative</text><text x='340.0' y='18' text-anchor='middle' font-size='14'>Fraction of the spectrum that is negative</text></svg>")

If the two colors track each other above, pinning has not introduced any real
indefiniteness -- both are sitting on the same compactness-driven noise floor.


## Resolvable rank: does pinning eat into it?

`audit_compliance_operator.jl`'s AUDIT 4 measured the resolvable pressure rank as a function
of `M`, `L`, `theta_c` for the free wall. A rank-one update can move rank by at most 1; here
that bound is checked directly rather than assumed.

In [5]:
cases = ((80, 80, 0.2), (80, 80, 0.4), (80, 80, 0.6), (160, 40, 0.4), (40, 160, 0.4))
rank_c = Int[]; rank_f = Int[]
for (Mv, Lv, tc) in cases
    q = Params(We=1.0958, Bo=0.017, Oh=0.006, M=Mv, L=Lv, N=8, b=6.0, h0=3.0, nq=160, wall=:clamped)
    for (wall, dest) in ((:clamped, rank_c), (:free, rank_f))
        A, _, _, _, w, x, om = compliance(q, tc, zeros(q.L + 1), delta; wall=wall)
        n = length(x)
        S = [w[i] * (-A[i, j]) / om[j] for i in 1:n, j in 1:n]
        sv = svdvals(Symmetric((S + S') / 2)); sv ./= sv[1]
        push!(dest, count(>(1e-8), sv))
    end
end
display(svgbars(["M=$M,L=$L\nθ=$tc" for (M, L, tc) in cases],
                 [(y=Float64.(rank_c), color="#1f77b4", label="clamped"), (y=Float64.(rank_f), color="#d62728", label="free")];
                 ylabel="resolvable rank (tol 1e-8)", title="Resolvable pressure rank: clamped vs free"))

HTML("<svg xmlns='http://www.w3.org/2000/svg' width='680' height='300' font-family='sans-serif' font-size='12'><rect width='680' height='300' fill='white'/><line x1='52' y1='248.0' x2='648.8' y2='248.0' stroke='black'/><text x='111.67999999999999' y='266' text-anchor='middle'>M=80,L=80\nθ=0.2</text><rect x='91.78666666666666' y='171.824' width='33.81866666666666' height='76.17599999999999' fill='#1f77b4'/><rect x='131.57333333333332' y='171.824' width='33.81866666666666' height='76.17599999999999' fill='#d62728'/><text x='231.03999999999996' y='266' text-anchor='middle'>M=80,L=80\nθ=0.4</text><rect x='211.14666666666665' y='129.50399999999996' width='33.81866666666666' height='118.49600000000004' fill='#1f77b4'/><rect x='250.9333333333333' y='129.50399999999996' width='33.81866666666666' height='118.49600000000004' fill='#d62728'/><text x='350.4' y='266' text-anchor='middle'>M=80,L=80\nθ=0.6</text><rect x='330.50666666666666' y='78.71999999999997' width='33.81866666666666' height='169.28000000000003' fill='#1f77b4'/><rect x='370.2933333333333' y='78.71999999999997' width='33.81866666666666' height='169.28000000000003' fill='#d62728'/><text x='469.75999999999993' y='266' text-anchor='middle'>M=160,L=40\nθ=0.4</text><rect x='449.86666666666656' y='121.03999999999999' width='33.81866666666666' height='126.96000000000001' fill='#1f77b4'/><rect x='489.65333333333325' y='121.03999999999999' width='33.81866666666666' height='126.96000000000001' fill='#d62728'/><text x='589.1199999999999' y='266' text-anchor='middle'>M=40,L=160\nθ=0.4</text><rect x='569.2266666666666' y='36.39999999999998' width='33.81866666666666' height='211.60000000000002' fill='#1f77b4'/><rect x='609.0133333333333' y='36.39999999999998' width='33.81866666666666' height='211.60000000000002' fill='#d62728'/><line x1='490' y1='40.0' x2='515' y2='40.0' stroke='#1f77b4' stroke-width='8'/><text x='521' y='44.0'>clamped</text><line x1='490' y1='54.0' x2='515' y2='54.0' stroke='#d62728' stroke-width='8'/><text x='521' y='58.0'>free</text><text x='14' y='150.0' text-anchor='middle' transform='rotate(-90 14 150.0)'>resolvable rank (tol 1e-8)</text><text x='340.0' y='18' text-anchor='middle' font-size='14'>Resolvable pressure rank: clamped vs free</text></svg>")

## Pinning constraint sanity check

Independent of the operator-assembly algebra above: build an arbitrary smooth pressure sample,
form `a_m` exactly the way `unpack_state` does (`kappa .* c_m`, then `apply_clamp`), and check
`sum_m a_m * j0kb_m = 0` (the actual volume-conservation constraint) to machine precision.

In [6]:
let theta_c = 0.3, seed_p = p
    xc = cos(theta_c)
    s, wq = gauss_legendre_nodes(seed_p.nq)
    x = @. xc + (1 + s) * (1 - xc) / 2
    om = @. wq * (1 - xc) / 2
    r = @. sqrt(1 - x^2)
    w = @. x
    pvec = sin.(3 .* x) .+ 0.5
    a = 1.5
    kappa_m = [(-2 * delta^2 * seed_p.k[m+1] * tanh(seed_p.k[m+1] * seed_p.h0)) /
               (a * (a + 4 * delta * seed_p.Oh * seed_p.k[m+1]^2) +
                delta^2 * (seed_p.k[m+1]^2 + seed_p.Bo) * seed_p.k[m+1] * tanh(seed_p.k[m+1] * seed_p.h0))
               for m in 0:seed_p.M]
    cm = [seed_p.bath_norm[m+1] * sum(pvec[i] * besselj0(seed_p.k[m+1] * r[i]) * w[i] * om[i] for i in eachindex(x))
          for m in 0:seed_p.M]
    am = apply_clamp(kappa_m .* cm, kappa_m, seed_p)
    @printf("sum_m a_m * j0kb_m = %.3e (machine zero expected)\n", sum(am .* seed_p.j0kb))
end

sum_m a_m * j0kb_m = -6.617e-24 (machine zero expected)


## Verdict

Volume-conserving pinning (`wall=:clamped`) leaves the Signorini structure established for
the free wall intact: the total operator asymmetry, the location and scale of the (compactness-
driven, not pinning-driven) near-zero eigenvalues, and the resolvable pressure rank all match
the free-wall values to within the rank-one correction's negligible practical effect. The
droplet-block projection mismatch already documented for `:free` remains the only real source
of asymmetry; pinning does not introduce a second one, and does not measurably change the
solver's effective conditioning story from `derivations/paper-formulation.tex` §6.5.